# ISSR Trust AI - Behavioral Analysis Notebook

This notebook analyzes experimental data from the trust calibration study to evaluate how different AI interface cues affect user trust and decision-making behavior.

## Analysis Overview
- Reliance rate by condition (Accept decisions)
- Override rate by condition (Reject decisions)
- Response latency analysis
- Statistical comparison between conditions

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import json

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

## Load Experimental Data

Loading data from both CSV and JSON export files.

In [ ]:
# Load data from CSV
try:
    df = pd.read_csv('logs.csv')
    print(f"✓ Successfully loaded {len(df)} trials from logs.csv")
except FileNotFoundError:
    print("CSV file not found. Trying JSON...")
    df = None

# Load data from JSON as backup
if df is None or len(df) == 0:
    try:
        with open('logs.json', 'r') as f:
            data = json.load(f)
        df = pd.DataFrame(data)
        print(f"✓ Successfully loaded {len(df)} trials from logs.json")
    except FileNotFoundError:
        print("❌ No data files found. Please run the experiment first.")
        df = None

# Display basic info
if df is not None and len(df) > 0:
    print(f"\nDataset shape: {df.shape[0]} trials × {df.shape[1]} variables")
    print("\nColumn names:", list(df.columns))
    print("\nFirst few rows:")
    display(df.head())

## Data Quality Check

Verify data completeness and check for any issues.

In [ ]:
if df is not None and len(df) > 0:
    # Check for missing values
    print("Missing Values:")
    print(df.isnull().sum())
    
    # Check data types
    print("\nData Types:")
    print(df.dtypes)
    
    # Summary statistics for latency
    print("\nLatency Statistics (ms):")
    print(df['latency_ms'].describe())
    
    # Condition distribution
    print("\nCondition Distribution:")
    print(df['condition'].value_counts())
    
    # Decision distribution
    print("\nDecision Distribution:")
    print(df['decision'].value_counts())

## Primary Analysis: Reliance & Override Rates

### Research Questions:
1. Do participants show different reliance rates based on AI tone (formal vs. conversational)?
2. Are override behaviors influenced by humanlike cues?

In [ ]:
if df is not None and len(df) > 0:
    # Calculate overall rates
    total_trials = len(df)
    overall_reliance = (df['decision'] == 'Accept').sum() / total_trials * 100
    overall_override = (df['decision'] == 'Reject').sum() / total_trials * 100
    
    print("=== OVERALL BEHAVIORAL METRICS ===")
    print(f"Total Trials: {total_trials}")
    print(f"Overall Reliance Rate: {overall_reliance:.1f}%")
    print(f"Overall Override Rate: {overall_override:.1f}%")
    
    # Calculate rates by condition
    print("\n=== METRICS BY CONDITION ===")
    
    for condition in sorted(df['condition'].unique()):
        cond_data = df[df['condition'] == condition]
        n_trials = len(cond_data)
        reliance_rate = (cond_data['decision'] == 'Accept').sum() / n_trials * 100
        override_rate = (cond_data['decision'] == 'Reject').sum() / n_trials * 100
        
        cond_name = "Formal/Technical (System-8)" if condition == "A" else "Conversational/Humanlike (Assistant Sarah)"
        print(f"\nCondition {condition} - {cond_name}:")
        print(f"  Trials: {n_trials}")
        print(f"  Reliance Rate: {reliance_rate:.1f}%")
        print(f"  Override Rate: {override_rate:.1f}%")

## Visualization: Decision Distribution

In [ ]:
if df is not None and len(df) > 0:
    # Create figure with subplots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Overall decision distribution
    decision_counts = df['decision'].value_counts()
    colors = ['#2ecc71' if d == 'Accept' else '#e74c3c' for d in decision_counts.index]
    
    axes[0].bar(decision_counts.index, decision_counts.values, color=colors, edgecolor='black', linewidth=1.5)
    axes[0].set_xlabel('Decision', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Count', fontsize=12, fontweight='bold')
    axes[0].set_title('Overall Decision Distribution', fontsize=14, fontweight='bold')
    
    # Add value labels
    for i, v in enumerate(decision_counts.values):
        axes[0].text(i, v + 0.1, str(v), ha='center', va='bottom', fontsize=11)
    
    # Plot 2: Decision distribution by condition
    decision_by_condition = pd.crosstab(df['condition'], df['decision'])
    decision_by_condition.plot(kind='bar', ax=axes[1], color=['#e74c3c', '#2ecc71'], 
                               edgecolor='black', linewidth=1.2, rot=0)
    axes[1].set_xlabel('Condition', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Count', fontsize=12, fontweight='bold')
    axes[1].set_title('Decisions by Experimental Condition', fontsize=14, fontweight='bold')
    axes[1].legend(['Reject', 'Accept'], title='Decision')
    
    # Add condition labels
    condition_labels = ['A: Formal/Technical', 'B: Conversational/Humanlike']
    axes[1].set_xticklabels(condition_labels, rotation=0)
    
    plt.tight_layout()
    plt.savefig('decision_distribution.png', dpi=300, bbox_inches='tight')
    print("✓ Saved decision distribution plot to 'decision_distribution.png'")
    plt.show()

## Latency Analysis

Analyzing response times to understand decision confidence and deliberation.

In [ ]:
if df is not None and len(df) > 0:
    print("=== RESPONSE LATENCY ANALYSIS ===\n")
    
    # Overall latency
    mean_latency = df['latency_ms'].mean()
    median_latency = df['latency_ms'].median()
    std_latency = df['latency_ms'].std()
    
    print("Overall Latency:")
    print(f"  Mean: {mean_latency:.0f} ms ({mean_latency/1000:.2f} seconds)")
    print(f"  Median: {median_latency:.0f} ms ({median_latency/1000:.2f} seconds)")
    print(f"  Std Dev: {std_latency:.0f} ms")
    
    # Latency by condition
    print("\nLatency by Condition:")
    for condition in sorted(df['condition'].unique()):
        cond_data = df[df['condition'] == condition]
        cond_mean = cond_data['latency_ms'].mean()
        cond_median = cond_data['latency_ms'].median()
        cond_std = cond_data['latency_ms'].std()
        
        cond_name = "Formal/Technical (System-8)" if condition == "A" else "Conversational/Humanlike (Assistant Sarah)"
        print(f"\nCondition {condition} - {cond_name}:")
        print(f"  Mean: {cond_mean:.0f} ms ({cond_mean/1000:.2f} s)")
        print(f"  Median: {cond_median:.0f} ms ({cond_median/1000:.2f} s)")
        print(f"  Std Dev: {cond_std:.0f} ms")
    
    # Latency by decision
    print("\nLatency by Decision:")
    for decision in ['Accept', 'Reject']:
        dec_data = df[df['decision'] == decision]
        if len(dec_data) > 0:
            dec_mean = dec_data['latency_ms'].mean()
            dec_median = dec_data['latency_ms'].median()
            print(f"{decision}: Mean = {dec_mean:.0f} ms, Median = {dec_median:.0f} ms")

## Visualization: Latency Comparison

In [ ]:
if df is not None and len(df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Box plot of latency by condition
    df.boxplot(column='latency_ms', by='condition', ax=axes[0], return_type='dict')
    axes[0].set_xlabel('Condition', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Response Latency (ms)', fontsize=12, fontweight='bold')
    axes[0].set_title('Response Latency Distribution by Condition', fontsize=14, fontweight='bold')
    
    # Customize x-axis labels
    axes[0].set_xticklabels(['A: Formal', 'B: Conversational'])
    
    # Plot 2: Bar chart of mean latency by condition
    mean_latencies = df.groupby('condition')['latency_ms'].mean()
    colors = ['#3498db' if c == 'A' else '#9b59b6' for c in mean_latencies.index]
    
    axes[1].bar(mean_latencies.index, mean_latencies.values, color=colors, 
                edgecolor='black', linewidth=1.5, yerr=df.groupby('condition')['latency_ms'].std())
    axes[1].set_xlabel('Condition', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Mean Response Latency (ms)', fontsize=12, fontweight='bold')
    axes[1].set_title('Mean Response Latency by Condition', fontsize=14, fontweight='bold')
    axes[1].set_xticklabels(['A: Formal', 'B: Conversational'])
    
    # Add value labels
    for i, v in enumerate(mean_latencies.values):
        axes[1].text(i, v + 100, f'{v:.0f}', ha='center', va='bottom', fontsize=11)
    
    plt.tight_layout()
    plt.savefig('latency_analysis.png', dpi=300, bbox_inches='tight')
    print("✓ Saved latency analysis plot to 'latency_analysis.png'")
    plt.show()

## Statistical Significance Testing

Testing whether observed differences between conditions are statistically significant.

In [ ]:
if df is not None and len(df) >= 10:
    print("=== STATISTICAL TESTS ===\n")
    
    # Chi-square test for independence (decision ~ condition)
    contingency_table = pd.crosstab(df['condition'], df['decision'])
    
    if contingency_table.min().min() >= 5:  # Chi-square assumption: expected freq >= 5
        chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)
        
        print("Chi-Square Test: Decision Independence by Condition")
        print(f"  Chi-square statistic: {chi2:.3f}")
        print(f"  Degrees of freedom: {dof}")
        print(f"  p-value: {p_value:.4f}")
        
        if p_value < 0.05:
            print("  ✓ Result: SIGNIFICANT (p < 0.05)")
            print("  Interpretation: Condition significantly affects decision behavior")
        else:
            print("  ✗ Result: NOT SIGNIFICANT (p ≥ 0.05)")
            print("  Interpretation: No significant effect of condition on decision behavior")
    else:
        print("⚠ Chi-square test assumptions not met (expected frequencies < 5)")
        print("  Need more data for reliable statistical testing")
    
    # T-test for latency differences (if enough data)
    cond_a_latency = df[df['condition'] == 'A']['latency_ms']
    cond_b_latency = df[df['condition'] == 'B']['latency_ms']
    
    if len(cond_a_latency) >= 5 and len(cond_b_latency) >= 5:
        t_stat, p_value_latency = stats.ttest_ind(cond_a_latency, cond_b_latency, equal_var=False)
        
        print(f"\nIndependent Samples T-Test: Latency Difference")
        print(f"  t-statistic: {t_stat:.3f}")
        print(f"  p-value: {p_value_latency:.4f}")
        
        if p_value_latency < 0.05:
            print("  ✓ Result: SIGNIFICANT (p < 0.05)")
            print("  Interpretation: Conditions have significantly different response times")
        else:
            print("  ✗ Result: NOT SIGNIFICANT (p ≥ 0.05)")
            print("  Interpretation: No significant difference in response times")
else:
    print("⚠ Insufficient data for statistical tests (need at least 10 trials)")
    print("  Collect more experimental data to enable inferential statistics")

## Summary Report

In [ ]:
if df is not None and len(df) > 0:
    print("="*60)
    print("EXPERIMENTAL ANALYSIS SUMMARY")
    print("="*60)
    
    print(f"\n📊 Dataset: {len(df)} total trials")
    print(f"   - Condition A (Formal/Technical): {(df['condition']=='A').sum()} trials")
    print(f"   - Condition B (Conversational): {(df['condition']=='B').sum()} trials")
    
    print(f"\n🎯 Key Findings:")
    
    # Reliance rates
    reliance_a = (df[df['condition']=='A']['decision']=='Accept').sum() / len(df[df['condition']=='A']) * 100
    reliance_b = (df[df['condition']=='B']['decision']=='Accept').sum() / len(df[df['condition']=='B']) * 100
    
    print(f"   • Reliance Rate - Condition A: {reliance_a:.1f}%")
    print(f"   • Reliance Rate - Condition B: {reliance_b:.1f}%")
    print(f"   • Difference: {abs(reliance_a - reliance_b):.1f} percentage points")
    
    # Latency
    latency_a = df[df['condition']=='A']['latency_ms'].mean()
    latency_b = df[df['condition']=='B']['latency_ms'].mean()
    
    print(f"\n⏱️  Response Times:")
    print(f"   • Mean Latency - Condition A: {latency_a:.0f} ms ({latency_a/1000:.2f}s)")
    print(f"   • Mean Latency - Condition B: {latency_b:.0f} ms ({latency_b/1000:.2f}s)")
    
    print(f"\n💡 Interpretation:")
    if reliance_b > reliance_a:
        print(f"   → Humanlike/conversational cues INCREASED trust/reliance")
    elif reliance_a > reliance_b:
        print(f"   → Formal/technical cues INCREASED trust/reliance")
    else:
        print(f"   → No difference in reliance between conditions")
    
    print("\n" + "="*60)
    print("Analysis complete! Check generated PNG files for visualizations.")
    print("="*60)